# Lesson 6: Tool Calling and Action Guardrails

## Agentic AI Learning Project

This notebook documents the implementation and testing of tool calling and action guardrails using the Gemini API.

### Topics Covered

- Creating tool functions
- Registering available tools
- Tool calling without arguments
- Tool calling with arguments
- Extracting tool names and arguments
- Manual tool execution
- Action guardrails
- Allowing safe actions
- Blocking unauthorized actions

---

## Lesson Objective

The objective of this lesson is to understand how an AI agent can interact with external tools and how action guardrails can control which tools are allowed to execute.

In this lesson, we practiced:

- Defining Python functions as tools
- Allowing Gemini to select an appropriate tool
- Handling tools with and without arguments
- Extracting the selected tool name and arguments
- Executing selected tools manually
- Creating an allowlist of permitted actions
- Blocking actions that are not authorized

## 1. Dataset Setup

In [1]:
import pandas as pd

data = {
    'region': ['north', 'south', 'east', 'west'],
    'sales': [10000, 20000, 15000, 12000]
}

df = pd.DataFrame(data)

df

,region,sales
0,north,10000
1,south,20000
2,east,15000
3,west,12000


## 2. Tool Functions

In [2]:
def find_top_sales_region():
    """Find the region with the highest sales."""

    top_region = df.groupby('region')['sales'].sum().idxmax()
    top_sales = df.groupby('region')['sales'].sum().max()

    return {
        'region': top_region,
        'sales': int(top_sales)
    }


def calculate_sales_by_region(region : str):
    """Calculate total sales for a specific region."""

    sales = df[df['region'].str.lower() == region.lower()]['sales'].sum()

    return int(sales)


def calculate_total_sales():
    """Calculate total sales."""

    return int(df['sales'].sum())

## 3. Tool Registry

In [3]:
available_tools = {
    'find_top_sales_region': find_top_sales_region,
    'calculate_sales_by_region': calculate_sales_by_region,
    'calculate_total_sales': calculate_total_sales
}

## 4. Gemini API Setup

In [4]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

## 5. Basic Gemini Tool Calling

In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Which is the highest sales region?",
    config={
        "tools": [
            find_top_sales_region
        ],
        "automatic_function_calling": {
            "disable": True
        }
    }
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={},
          id='call_800580',
          name='find_top_sales_region'
        ),
        thought_signature=b'\x12\xff\x02\n\xfc\x02\x01\x11M2\x0f\xbb\xb6G\xd3\x06\xc8v\x00F\x82\xca\x05j\x07\x80z\x82\xc3\xea\x9a\x01`\xf3\x81\xac;V\x13\x13z\x14\xf4?\xa1-l^\xec\x9f\xdb\xed\xbcC\xd9\xf5?a\xb7r#t\x03>\xa2\x04/7\xb2FW\xb7\xa0\xb1\x7f\x16\xbf228\xd3\xdaB\xbb\xa8O9\x1e`\x7f\xf3\x93m\x87\x9eo\x15...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.6-flash' prompt_feedback=None response_id='veGNavriC73Vz7IPs9LdkQU' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=14,
  prompt_token_count=33,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=33
 

## 6. Extract the Function Call

In [ ]:
function_call = response.candidates[0].content.parts[0].function_call

tool_name = function_call.name
arguments = dict(function_call.args)

print("Tool:", tool_name)
print("Arguments:", arguments)

Tool: find_top_sales_region
Arguments: {}


## 7. Manual Tool Execution

In [5]:
def execute_tool(tool_name, arguments=None):

    tool_function = available_tools[tool_name]

    if arguments is None:
        return tool_function()

    return tool_function(**arguments)

In [ ]:
result = execute_tool(tool_name, arguments)

print("Tool:", tool_name)
print("Arguments:", arguments)
print("Result:", result)

Tool: find_top_sales_region
Arguments: {}
Result: {'region': 'south', 'sales': 20000}


## 8. Tool Calling With Arguments

In [8]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Calculate sales for the north region.",
    config={
        "tools": [
            calculate_sales_by_region
        ],
        "automatic_function_calling": {
            "disable": True
        }
    }
)

In [9]:
function_call = response.candidates[0].content.parts[0].function_call

tool_name = function_call.name
arguments = dict(function_call.args)

print("Tool:", tool_name)
print("Arguments:", arguments)

Tool: calculate_sales_by_region
Arguments: {'region': 'north'}


In [ ]:
result = execute_tool(tool_name, arguments)

print("Result:", result)

Result: 10000


## 9. Action Guardrails

In [6]:
allowed_tools = {
    'find_top_sales_region': find_top_sales_region,
    'calculate_sales_by_region': calculate_sales_by_region,
    'calculate_total_sales': calculate_total_sales
}

In [7]:
def execute_tools(tool_name, arguments=None):

    if tool_name not in allowed_tools:
        return {
            'status': 'blocked',
            'message': f"action '{tool_name}' is not allowed."
        }

    tool_function = allowed_tools[tool_name]

    if arguments is None:
        return tool_function()

    return tool_function(**arguments)

## 10. Blocked Action Test

In [ ]:
result = execute_tools('delete_dataset')

print(result)

{'status': 'blocked', 'message': "action 'delete_dataset' is not allowed."}


## Key Learnings

- AI agents can use Python functions as tools.
- Gemini can select tools and provide the required arguments.
- Tools can work with or without arguments.
- Function arguments should have clear type annotations for tool calling.
- Tool selection and tool execution are separate steps.
- A tool registry connects tool names to Python functions.
- Action guardrails use an allowlist to control which actions can execute.
- Unauthorized actions can be blocked before execution.